## Running this notebook

Install the project with the notebook extra before opening this file. Imports
use the installed package and do not change the working directory. Set
`OVERSCREENING_OUTPUT_DIR` to choose a directory for saved files.

The original scientific parameter values and sweep sizes are retained. Several
cells are deliberately expensive or demonstrate a failed solve; they are not
part of the bounded CI examples. Inspect convergence flags before interpreting
results. The optional fsolve initializer and native adjoint fallback have separate
controls; reference continuation remains a third, independent control. After a failed
physical Newton solve it computes a screened linear reference, then follows
the full residual from `q=1` to the physical endpoint `q=0`, retaining accepted
states and reducing the parameter step after failed stages. Source callbacks
remain physical and independent of `q`.
For bounded NumPy/mpmath demonstrations, run the scripts in `examples/`.

Ordinary BSK examples construct a fresh immutable `BSKProblem`; there are no
parameter setters in that interface. The initial problem defaults retain
sigma=10, mu=0, gamma=0.5, delta=50 and wall value 100. Every later parameter
block specifies its physical values and boundary data explicitly. Finite
segments use `domain` and both endpoint boundary rows. Unsupported boundary experiments have a separate opt-in research notebook.

Live solver diagnostics use `verbosity=2` in this notebook. Set it to `0`
for quiet solves, `1` for phase summaries, or `3` for trial details.


This file contains a test for different values of $\mu$ for different values of $L$.
In the end it provides graphs that deminstrate the diference of solutions for different $\mu$ values
as well as show relative error (in %) in $l_2$ norm between different values of $L$.

In [ ]:
# Install the project first: python -m pip install -e ".[notebook]"
from pathlib import Path
import os
from matplotlib import pyplot as plt
import numpy as np
import mpmath as mp
from overscreening_breakdown import (
    BSKProblem, MpmathArithmetic, solve, load_solution,
    ContinuationOptions, ScreenedLinearReference,
)
from overscreening_breakdown.rational_mapping import RationalHalfLineMap
OUTPUT_DIR = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
%matplotlib inline

def display_values(function, coordinates):
    # Float conversion is confined to plot data.
    return np.array([float(function(value)) for value in coordinates])

def mapped_values(result, angles):
    a = result.arithmetic
    if result.domain[1] == a.inf:
        mapping = RationalHalfLineMap(a, L=result.L)
        coordinates = [mapping.from_angle(a.scalar(value)) for value in angles]
    else:
        left, right = result.domain
        coordinates = [(a.cos(a.scalar(value))*(right-left)+(right+left))/2 for value in angles]
    return display_values(result.u, coordinates)

def run_collocation(problem, N, *, L=1, tolerance="1e-10", method="newton",
                    max_iterations=50, globalization=True, continuation=None, initial=None,
                    use_mpmath=False, precision=100, initializer=None,
                    initializer_tolerance="0.1", initializer_max_evaluations=200,
                    newton_fallback=None, adjoint_max_steps=20, verbosity=2,
                    progress_callback=None, progress_interval=1.0):
    if initial is not None and hasattr(initial, "coefficients"):
        initial = (initial.coefficients if initial.N == N
                   else lambda x, a, previous=initial: previous.u(x))
    return solve(problem, N, solver="collocation", L=L, tolerance=tolerance,
                 method=method, max_iterations=max_iterations,
                 use_globalization=globalization, continuation=continuation,
                 initial_guess=initial,
                 use_mpmath=use_mpmath, precision=precision,
                 initializer=initializer, initializer_tolerance=initializer_tolerance,
                 initializer_max_evaluations=initializer_max_evaluations,
                 newton_fallback=newton_fallback, adjoint_max_steps=adjoint_max_steps,
                 verbosity=verbosity, progress_callback=progress_callback,
                 progress_interval=progress_interval)


In [ ]:
sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 0.1
u0xxx = 0
def some_func(x, a):
    return u0 * a.exp(-x ** 2)
problem = BSKProblem()
N = 100
domain = [0,np.inf]
tolerance = 1.0e-12
use_method = "newton"
initializer = "fsolve"
Ls = [4,5,6]
mu_all = [0.35, 0.36, 0.37, 0.38, 0.39]
rhs_all = []
x = np.arange(0.0, 50.0, 0.1)
previous = None
for L in Ls:
    rhs_all_mu = []
    solution_all = []
    for mu_l in mu_all:
        print("L=", L, "mu=", mu_l)
        problem = BSKProblem(sigma=sigma, mu=mu_l, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)
        current = run_collocation(problem, N, L=L, tolerance=tolerance, method=use_method,
                                   initializer=initializer, initial=previous)
        previous = current
        native_x = [current.arithmetic.scalar(value) for value in x]
        # Use the unscaled physical source from the bound snapshot.
        rhs_l = np.array([float(current.physical.physical_rhs(point, current.u(point)))
                          for point in native_x])
        rhs_all_mu.append(rhs_l)
        solution_all.append(display_values(current.u, native_x))
    rhs_all.append(rhs_all_mu)
plt.title(r"solution on part of $\mathbb{R}^+$")
for values, mu_l in zip(solution_all, mu_all):
    plt.plot(x, values, label=r"$\mu$={}".format(mu_l))
plt.legend()
plt.show()
plt.title(r"physical RHS on part of $\mathbb{R}^+$")
for rhs_all_L, L in zip(rhs_all, Ls):
    for rhs, mu_l in zip(rhs_all_L, mu_all):
        plt.plot(x, rhs, label=r"$\mu$={}, L={}".format(mu_l, L))
plt.legend()
plt.show()


In [ ]:
#simple integration for a test. More advanced quadartures can be used instead.

def integrate_rhs(rhs, dx):
    i_val = sum(rhs)*dx
    return i_val;
    
        
def integrate_rhs_up_to_x(rhs, dx, all_x, to_x):
    mask = all_x<=to_x
    return sum(rhs*mask)*dx

def integrate_rhs_up_to_x_all(rhs, dx, all_x, to_x):
    res = []
    for x_p in to_x:
        mask = all_x<=x_p
        res.append(sum(rhs*mask)*dx)
    
    return res

In [ ]:
dx = 0.1/50.0
mask = np.tril(np.ones([len(x),len(x)]))
xx = mask*x
res_all = []
for rhs_all_L in rhs_all:
    res_all_L = []
    for rhs in rhs_all_L:
        res = integrate_rhs_up_to_x_all(rhs, dx, x, xx)/integrate_rhs(rhs, dx)
        res_all_L.append(res)
    res_all.append(res_all_L)

In [ ]:
#plotting the results.

legend = []
for res_all_L, L in zip(res_all, Ls):
    for res, mu in zip(res_all_L,  mu_all):
        plt.plot(x, res)
        legend.append(r"$\mu$ = {}, L = {}".format(mu, L))
plt.legend(legend)
plt.show()

In [ ]:
#printing output integral error.

print("relative L2 error in % between L=4,6 and L=5,6")
for res_4, res_5, res_6, mu in zip(res_all[0], res_all[1], res_all[2], mu_all):
    l2norm6 = np.sqrt(sum((res_6)**2))
    err65 = np.sqrt(sum((res_6-res_5)**2))/l2norm6
    err64 = np.sqrt(sum((res_6-res_4)**2))/l2norm6
    print("mu = ", mu, " err:", err65*100.0, err64*100.0)